# This file consists of the main vector embedding which will be done by the langchain

In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
# from langchain_openai import OpenAIEmbeddings we will use the sentence_transformers model for embedding the text. Cause me gareeb hu and i don't have Tokens in openai api key.


In [11]:
import torch
from sentence_transformers import SentenceTransformer #The syntax would be a little different from openai embedding will try to explain the code.
from langchain_huggingface import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
# For models instead of using openai we will use the groq.
from dotenv import load_dotenv
load_dotenv(override=True)
import os
groq_api_key = os.getenv("GROQ_API_KEY")
if load_dotenv():
        print(f"Groq API key loaded successfully and starts with {groq_api_key[:5]}")
else:
        print("No API KEy found")

Groq API key loaded successfully and starts with gsk_l


In [1]:
import pandas as pd
books = pd.read_csv("books_cleaned.csv")

In [ ]:
books #This is the clean dataset of books that we will use.

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critic...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web - A Novel,9780002261982 A new 'Christie for Christmas' ...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroin..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lo...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. L..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6139,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a m...
6140,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Pass...
6141,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that - Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless...
6142,9789027712059,9027712050,The Berlin Phenomenology,Georg Wilhelm Friedrich Hegel,History,http://books.google.com/books/content?id=Vy7Sk...,Since the three volume edition ofHegel's Philo...,1981.0,0.00,210.0,0.0,The Berlin Phenomenology,9789027712059 Since the three volume edition ...


In [3]:
books["tagged_description"]

0       9780002005883  A NOVEL THAT READERS and critic...
1       9780002261982  A new 'Christie for Christmas' ...
2       9780006178736  A memorable, mesmerizing heroin...
3       9780006280897  Lewis' work on the nature of lo...
4       9780006280934  "In The Problem of Pain, C.S. L...
                              ...                        
6139    9788173031014  This book tells the tale of a m...
6140    9788179921623  Wisdom to Create a Life of Pass...
6141    9788185300535  This collection of the timeless...
6142    9789027712059  Since the three volume edition ...
6143    9789042003408  This is a jubilant and rewardin...
Name: tagged_description, Length: 6144, dtype: str

In [4]:
books["tagged_description"].to_csv("tagged_description.txt",index=False,sep="\n",header=False)

In [ ]:
raw_documents = TextLoader("tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=0.5,chunk_overlap=0,separator="\n")
documents = text_splitter.split_documents(raw_documents)

In [10]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883  A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, G

In [ ]:
db_books = Chroma.from_documents(
    documents,
    embedding=embedding_model,
)
# so we have created a vector database using chroma and the embedding model is the sentence transformer model. We can now use this vector database to perform similarity search and retrieve relevant documents based on a query.

In [14]:
query = "What are some books about love and relationships?"
docs = db_books.similarity_search(query, k=10) # k is the number of the results which we want to retrieve. We can change the numbers.
docs

[Document(id='0513ab8d-f064-4c32-9c38-0cd564a9f328', metadata={'source': 'tagged_description.txt'}, page_content='9780060574215  Rediscover the most famous relationship book ever published Once upon a time Martians and Venusians met, fell in love, and had happy relationships together because they respected and accepted their differences. Then they came to Earth and amnesia set in: they forgot they were from different planets. Based on years of successful counseling of couples and individuals, Men Are from Mars, Women Are from Venus has helped millions of couples transform their relationships. Now viewed as a modern classic, this phenomenal book has helped men and women realize how different they can be in their communication styles, their emotional needs, and their modes of behavior—and offers the secrets of communicating without conflicts, allowing couples to give intimacy every chance to grow.'),
 Document(id='bdd02152-e012-4ebf-b198-cfe19e7a7928', metadata={'source': 'tagged_descrip

In [15]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())] # so we can see that the first result is the book "The Notebook" which is a book about love and relationships. We can also see that the second result is the book "Pride and Prejudice" which is also a book about love and relationships. So we can see that the vector search is working correctly and retrieving relevant documents based on the query.

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
175,9780060574215,0060574216,"Men Are from Mars, Women Are from Venus",John Gray,Family & Relationships,http://books.google.com/books/content?id=MUw_d...,Rediscover the most famous relationship book e...,2004.0,3.54,368.0,126108.0,"Men Are from Mars, Women Are from Venus - The ...",9780060574215 Rediscover the most famous rela...


In [ ]:
def retrieve_semantic_recommendations(
        query:str,
        top_k: int = 10
) -> pd.DataFrame:
        recs = db_books.similarity_search(query, k =50)
        books_list = []
        for i in range(0,len(recs)):
                books_list += [int(recs[i].page_content.strip('"').split()[0])]
        return books[books["isbn13"].isin(books_list)].head(top_k)
# this function takes a query and the number of recommendations we want to retrieve and returns a dataframe of the recommended books based on the query. We can use this function to get recommendations for any query we want. For example, if we want to get recommendations for books about love and relationships.

In [17]:
#  we can call the abive function like this:
retrieve_semantic_recommendations("What are some books about love and relationships?")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lo...
78,9780007189953,0007189958,Where Rainbows End,Cecelia Ahern,Friendship,http://books.google.com/books/content?id=PA7t6...,The new warm and absorbing story from the auth...,2004.0,3.94,454.0,642.0,Where Rainbows End,9780007189953 The new warm and absorbing stor...
114,9780060175641,0060175648,Identity,Milan Kundera,Fiction,http://books.google.com/books/content?id=D30Ex...,Milan Kundera's Identity translated from the F...,1998.0,3.68,176.0,260.0,Identity - A Novel,9780060175641 Milan Kundera's Identity transl...
175,9780060574215,0060574216,"Men Are from Mars, Women Are from Venus",John Gray,Family & Relationships,http://books.google.com/books/content?id=MUw_d...,Rediscover the most famous relationship book e...,2004.0,3.54,368.0,126108.0,"Men Are from Mars, Women Are from Venus - The ...",9780060574215 Rediscover the most famous rela...
230,9780060753634,0060753633,Mating in Captivity,Esther Perel,Psychology,http://books.google.com/books/content?id=-HIhM...,A guide for loving couples who are looking to ...,2006.0,4.13,272.0,7699.0,Mating in Captivity - Reconciling the Erotic a...,9780060753634 A guide for loving couples who ...
234,9780060760441,0060760443,The Reading Group,Elizabeth Noble,Fiction,http://books.google.com/books/content?id=IagWj...,The Reading Group follows the trials and tribu...,2005.0,3.34,429.0,6408.0,The Reading Group - A Novel,9780060760441 The Reading Group follows the t...
308,9780060916466,006091646X,The Dance of Intimacy,Harriet Lerner,Psychology,http://books.google.com/books/content?id=tTKc8...,"In The Dance of Intimacy, the bestselling auth...",1990.0,4.06,255.0,7128.0,The Dance of Intimacy - A Woman's Guide to Cou...,"9780060916466 In The Dance of Intimacy, the b..."
318,9780060925758,0060925752,Soul Mates,Thomas Moore,Psychology,http://books.google.com/books/content?id=7syEl...,This companion volume to Care of the Soul offe...,1994.0,4.00,288.0,4122.0,Soul Mates,9780060925758 This companion volume to Care o...
331,9780060930318,0060930314,Identity,Milan Kundera,Fiction,http://books.google.com/books/content?id=mXPU2...,There are situations in which we fail for a mo...,1999.0,3.68,168.0,13065.0,Identity - A Novel,9780060930318 There are situations in which w...
350,9780060956868,0060956860,Joy in the Morning,Betty Smith,Fiction,http://books.google.com/books/content?id=w2uMG...,The story of a young couple from Brooklyn who ...,2000.0,3.90,296.0,5559.0,Joy in the Morning,9780060956868 The story of a young couple fro...
